# nb01 — Empty-Annotation Baseline Fine-tune

Reproduces the host's `simple-fine-tuning-baseline` exactly:
fine-tune the poisoned RetinaNet on the 20 unlearn images with **empty annotations** for
20 iterations (lr=1e-4, batch=4), then infer on the 2 000 test images.

**Expected LB score:** ~223–224 (matches the public baseline cluster).

## 1. Install Detectron2

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

## 2. Imports

In [ ]:
import copy
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from tqdm import tqdm

## 3. Configuration

In [ ]:
# ── Competition data paths (handles both Kaggle mount styles) ──
def _find_comp_data():
    competitions = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
    standard     = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
    return competitions if competitions.exists() else standard

COMP_ROOT        = _find_comp_data()
POISONED_WEIGHTS = str(COMP_ROOT / "poisoned_model/poisoned_model.pth")
UNLEARN_DIR      = str(COMP_ROOT / "unlearn_set")
TEST_DIR         = str(COMP_ROOT / "test_set/test_set")
OUTPUT_DIR       = "/kaggle/working/unlearned"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"

# ── Model architecture (MUST match poisoned model training config) ──
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1

# ── Unlearning hyperparameters (matching the host baseline exactly) ──
UNLEARN_LR    = 1e-4
UNLEARN_ITERS = 20
BATCH_SIZE    = 4

# ── Inference ──
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024

print(f"Competition data root : {COMP_ROOT}")
print(f"Poisoned weights      : {POISONED_WEIGHTS}")
print(f"Unlearn dir           : {UNLEARN_DIR}")
print(f"Test dir              : {TEST_DIR}")

## 4. Register the unlearn dataset

Annotations are **discarded** (`"annotations": []`). Training on these images with empty
labels tells the model 'nothing of interest here' — suppressing the poisoned detections.

In [ ]:
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    dicts = [
        {
            "file_name":   str(Path(unlearn_dir) / im["file_name"]),
            "height":      im["height"],
            "width":       im["width"],
            "image_id":    im["id"],
            "annotations": [],  # empty = unlearning signal
        }
        for im in coco["images"]
    ]
    DatasetCatalog.register(UNLEARN_DATASET, lambda d=dicts: d)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered unlearn set: {len(dicts)} images (empty annotations)")

register_unlearn(UNLEARN_DIR)

## 5. Custom DatasetMapper and Trainer

In [ ]:
class UInt16DatasetMapper(DatasetMapper):
    """Reads 16-bit PNGs as float32 in [0, 255] and attaches empty instances (unlearning signal)."""

    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)
        dataset_dict["image"]     = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


class UnlearnTrainer(DefaultTrainer):
    """DefaultTrainer that keeps images with empty annotations (needed for the forget step)."""

    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

## 6. Inference helper

In [ ]:
def load_for_inference(path):
    """Load a 16-bit PNG and return float32 HWC array in [0, 255] with 3 channels."""
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

## 7. Build config and run unlearning

`MAX_ITER=20` means **20 gradient steps** (not epochs).  With `IMS_PER_BATCH=4` on 20
images that is ~4 effective passes through the unlearn set.

In [ ]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))

cfg.MODEL.WEIGHTS                        = POISONED_WEIGHTS
cfg.MODEL.RETINANET.NUM_CLASSES          = NUM_CLASSES
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
cfg.MODEL.ANCHOR_GENERATOR.SIZES         = ANCHOR_SIZES

cfg.DATASETS.TRAIN = (UNLEARN_DATASET,)
cfg.DATASETS.TEST  = ()

cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
cfg.SOLVER.BASE_LR         = UNLEARN_LR
cfg.SOLVER.MAX_ITER        = UNLEARN_ITERS
cfg.SOLVER.STEPS           = []

cfg.OUTPUT_DIR = OUTPUT_DIR
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

trainer = UnlearnTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()
print("Training complete.")

## 8. Inference on test set → submission.csv

In [ ]:
cfg.MODEL.WEIGHTS                      = str(Path(OUTPUT_DIR) / "model_final.pth")
cfg.MODEL.RETINANET.SCORE_THRESH_TEST  = CONF_THRESH
predictor = DefaultPredictor(cfg)

test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im  = load_for_inference(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([
            f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"
        ])

    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}  ({len(submission)} rows)")
submission.head(10)

## 9. Validate submission format

In [ ]:
# Hard assertions before submission
assert list(submission.columns) == ["id", "image_id", "prediction_string"], \
    f"Wrong columns: {list(submission.columns)}"
assert len(submission) == 2000, f"Expected 2000 rows, got {len(submission)}"
assert (submission["id"].values == list(range(2000))).all(), "id column is not 0..1999"

# Summary stats
empty_rows = (submission["prediction_string"].str.strip() == "").sum()
total_dets = submission["prediction_string"].apply(
    lambda s: len(s.split()) // 5 if isinstance(s, str) and s.strip() else 0
).sum()

print(f"✓ Submission valid: 2000 rows, correct columns.")
print(f"  Images with no detections : {empty_rows}")
print(f"  Total detections          : {total_dets}")